In [17]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [ ]:
import joblib

project_root = Path("/Users/alexgonzalez/Documents/NBA-Prop-Predictor")  # or Path.cwd()
min_path = project_root / "src/models/saved_models/min_quantile_xgb_2026-02-04.joblib"
ppm_path = project_root / "src/models/saved_models/ppm_quantile_xgb_2026-02-11.joblib"

min_bundle = joblib.load(min_path)
min_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

ppm_bundle = joblib.load(ppm_path)
ppm_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]


#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

### Helper Functions

In [45]:
from src.live import *

def get_rate_history(df, player, date, n_games=20, rate_col='PTS_PER_MIN'):
    pdf = df[(df['PLAYER_NAME'] == player) & (df['GAME_DATE'] < date)]
    pdf = pdf.sort_values(by='GAME_DATE', ascending=True)
    rate_history = pdf[rate_col].dropna().tail(n_games)
    return rate_history

def grab_player_last_game(df, features, player, date):
    pdf = df[(df['PLAYER_NAME'] == player) & (df['GAME_DATE'] == date)]
    if pdf.empty:
        return f"No data found for {player} on {date}"
    return pdf[features]

def build_sim_row(player, min_arr, ppm_arr, min_models, ppm_models, rate_history):
    """
    Build a row dict compatible with `run_pts_simulation` in src/live.py.
    Predicts q10/q50/q90 for both MIN and PPM and attaches rate history.
    """
    q10, q50, q90 = 'q_0.10', 'q_0.50', 'q_0.90'

    m10 = float(min_models[q10].predict(min_arr)[0])
    m50 = float(min_models[q50].predict(min_arr)[0])
    m90 = float(min_models[q90].predict(min_arr)[0])

    r10 = float(ppm_models[q10].predict(ppm_arr)[0])
    r50 = float(ppm_models[q50].predict(ppm_arr)[0])
    r90 = float(ppm_models[q90].predict(ppm_arr)[0])

    return {
        'PLAYER_NAME': player,
        'MARKET': 'PTS',
        'MIN_Q10': m10, 'MIN_Q50': m50, 'MIN_Q90': m90,
        'RATE_Q10': r10, 'RATE_Q50': r50, 'RATE_Q90': r90,
        'STAT_Q10': m10 * r10, 'STAT_Q50': m50 * r50, 'STAT_Q90': m90 * r90,
        'RATE_HISTORY': list(rate_history) if rate_history is not None else [],
    }


### Solo lookup

In [50]:
player = 'Shai Gilgeous-Alexander'
date = '2026-04-02'
last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
last_ppm = grab_player_last_game(pts_df, ppm_feature_names, player, date)

min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
ppm_arr = np.asarray(last_ppm, dtype=float).reshape(1, -1)

min_pred = min_models['q_0.50'].predict(min_arr)
ppm_pred = ppm_models['q_0.50'].predict(ppm_arr)
ppm_pred_q90 = ppm_models['q_0.90'].predict(ppm_arr)
rate_history = get_rate_history(pts_df, player, '2026-04-02').to_list()

print(f"predicted MIN: {min_pred[0]:.2f}, predicted PPM: {ppm_pred[0]:.2f}, predicted PTS: {min_pred[0] * ppm_pred[0]:.2f}, predicted PTS q90: {min_pred[0] * ppm_pred_q90[0]:.2f}")
print(f"rate history: {rate_history}")
last_ppm

predicted MIN: 35.36, predicted PPM: 0.92, predicted PTS: 32.54, predicted PTS q90: 40.35
rate history: [0.6818827540486789, 0.8094152672465925, 0.9549071618037136, 1.016949152542373, 0.7194244604316546, 1.0661401776900297, 0.8962804361898123, 0.7330827067669172, 0.7714285714285715, 0.8985879332477534, 0.8955223880597014, 0.6018054162487462, 1.1230697239120262, 0.7822685788787485, 1.2572027239392354, 0.7573149741824441, 0.8928571428571428, 0.8479366873940078, 0.8148483476686282, 1.1595394736842106]


,PLAYER_ENCODED,STARTING,TOP_USG_PLAYER_STARTING,LINEUP_USG_SHIFT,TEAM_MIN_RANK_L10,TEAM_USG_RANK_L10,PPM_SEASON_STD,PTS_PER_MIN_season_avg,PPM_ROLE_Z_SCORE,MIN_TREND,TS_PCT_X_USG_PCT,PTS_PER_MIN_X_OPP_PACE,PTS_PER_MIN_X_OPP_PTS_ALLOWED,FGA_PER_MIN_X_OPP_FG%_ALLOWED,FTA_PER_MIN_X_OPP_PFD_ALLOWED,3PM_PER_MIN_X_OPP_FG3%_ALLOWED
85254,131,1,1.0,0.0,1.0,1.0,0.17344,0.94,-0.080748,35.81,0.2176,92.340677,106.66821,0.26565,6.54395,0.01408


### Week lookup 

In [48]:
date = '2026-03-21'
N_SIMS = 10_000
PROB_THRESHOLD = 0.58  # only "recommend" a bet if sim prob on chosen side clears this

np.random.seed(42)

backtest_df = pd.read_csv(f'/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/backtest/historical_odds/{date}.csv')
backtest_df = backtest_df[backtest_df['prop'] == 'Points']

res = []

for player in backtest_df['player']:
    pdf_pts_mean = pts_df[(pts_df['PLAYER_NAME'] == player)]['PTS'].mean()
    pdf = pts_df[(pts_df['PLAYER_NAME'] == player) & (pts_df['GAME_DATE'] == date)]
    if pdf.empty:
        print(f"No data found for {player} on {date}")
        continue
    last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
    last_ppm = grab_player_last_game(pts_df, ppm_feature_names, player, date)
    if not isinstance(last_mins, pd.DataFrame) or not isinstance(last_ppm, pd.DataFrame):
        continue
    line = backtest_df[backtest_df['player'] == player]['line'].values[0]
    actual_pts = pdf['PTS'].values[0]
    actual_min = pdf['MIN'].values[0]

    min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
    ppm_arr = np.asarray(last_ppm, dtype=float).reshape(1, -1)

    rate_history = get_rate_history(pts_df, player, date, n_games=20).tolist()
    sim_row = build_sim_row(player, min_arr, ppm_arr, min_models, ppm_models, rate_history)

    sims = run_pts_simulation(sim_row, n_sims=N_SIMS)
    p_over = float(np.mean(sims > line))
    p_under = float(np.mean(sims < line))
    sim_mean = float(np.mean(sims))
    sim_p10, sim_p50, sim_p90 = np.percentile(sims, [10, 50, 90])

    pred = sim_row['STAT_Q50']
    side = 'Over' if p_over >= p_under else 'Under'
    p_side = p_over if side == 'Over' else p_under
    hit = (side == "Over" and actual_pts > line) or (side == "Under" and actual_pts < line)
    edge = abs(pred - line)
    reccomended_bet = 1 if p_side >= PROB_THRESHOLD else 0

    res.append({
        'player': player,
        'player_pts_mean': pdf_pts_mean,
        'pred points': round(pred, 2),
        'sim_mean': round(sim_mean, 2),
        'sim_p10': round(float(sim_p10), 2),
        'sim_p50': round(float(sim_p50), 2),
        'sim_p90': round(float(sim_p90), 2),
        'side': side,
        'line': line,
        'actual points': actual_pts,
        'p_over': round(p_over, 3),
        'p_under': round(p_under, 3),
        'p_side': round(p_side, 3),
        'edge': round(edge, 2),
        'hit': int(hit),
        'reccomended_bet': reccomended_bet
    })
res = pd.DataFrame(res)
print('-' * 100)
print(f"Total hit rate: {round(res['hit'].sum() / len(res), 3)}")
rec = res[res['reccomended_bet'] == 1]
if len(rec):
    print(f"Recommended bets (p_side >= {PROB_THRESHOLD}): {len(rec)} | hit rate: {round(rec['hit'].sum() / len(rec), 3)}")
res.head(20)

No data found for Kevin Porter Jr. on 2026-03-21
No data found for Bobby Portis Jr. on 2026-03-21
No data found for Kasparas Jakucionis on 2026-03-21
No data found for Moussa Diabate on 2026-03-21
No data found for GG Jackson II on 2026-03-21
----------------------------------------------------------------------------------------------------
Total hit rate: 0.463
Recommended bets (p_side >= 0.58): 48 | hit rate: 0.479


,player,player_pts_mean,pred points,sim_mean,sim_p10,sim_p50,sim_p90,side,line,actual points,p_over,p_under,p_side,edge,hit,reccomended_bet
0,Zion Williamson,23.005236,19.35,20.99,12.40,19.64,31.89,Under,20.5,25,0.457,0.543,0.543,1.15,0,0
1,Grant Williams,8.903382,6.14,7.13,0.11,6.76,13.59,Over,5.5,5,0.603,0.397,0.603,0.64,0,1
2,Jalen Green,20.529412,21.65,21.63,11.19,20.30,34.29,Under,23.5,24,0.376,0.624,0.624,1.85,0,1
3,Alperen Sengun,18.730769,19.27,20.44,12.06,18.49,32.03,Under,18.5,19,0.499,0.501,0.501,0.77,0,0
4,Olivier-Maxence Prosper,5.875862,14.53,16.45,8.35,15.67,26.29,Over,12.5,10,0.702,0.298,0.702,2.03,0,1
5,Jabari Smith Jr.,13.726644,15.53,17.88,10.58,17.44,26.07,Over,14.5,13,0.714,0.286,0.714,1.03,0,1
6,Jaden Hardy,8.378723,9.41,11.51,3.32,11.26,19.51,Under,11.5,10,0.483,0.517,0.517,2.09,1,0
7,Jordan Goodwin,7.087156,11.69,11.08,3.87,10.15,20.09,Under,10.5,11,0.480,0.520,0.520,1.19,0,0
8,Corey Kispert,11.454212,8.18,9.27,2.08,8.52,17.56,Over,7.5,8,0.565,0.435,0.565,0.68,1,0
9,Jared McCain,10.111111,9.03,12.71,5.32,11.89,21.29,Under,13.5,18,0.407,0.593,0.593,4.47,0,1


In [49]:
from pathlib import Path

ODDS_DIR = Path("/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/backtest/historical_odds")

N_SIMS = 10_000
PROB_THRESHOLD = 0.58

def backtest_one_date(date, *, pts_df, min_df, min_models, ppm_models,
                      min_feature_names, ppm_feature_names, odds_dir=ODDS_DIR,
                      n_sims=N_SIMS, prob_threshold=PROB_THRESHOLD, verbose=False):
    fp = odds_dir / f"{date}.csv"
    if not fp.exists():
        return pd.DataFrame()
    bt = pd.read_csv(fp)
    bt = bt[bt["prop"] == "Points"]
    rows = []

    for player in bt["player"]:
        pdf_pts_mean = pts_df[(pts_df["PLAYER_NAME"] == player)]["PTS"].mean()
        pdf = pts_df[(pts_df["PLAYER_NAME"] == player) & (pts_df["GAME_DATE"] == date)]
        if pdf.empty:
            if verbose:
                print(f"[{date}] no data: {player}")
            continue

        last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
        last_ppm  = grab_player_last_game(pts_df, ppm_feature_names, player, date)
        if not isinstance(last_mins, pd.DataFrame) or not isinstance(last_ppm, pd.DataFrame):
            continue

        line = bt.loc[bt["player"] == player, "line"].values[0]
        actual_pts = pdf["PTS"].values[0]

        min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
        ppm_arr = np.asarray(last_ppm,  dtype=float).reshape(1, -1)

        rate_history = get_rate_history(pts_df, player, date, n_games=20).tolist()
        sim_row = build_sim_row(player, min_arr, ppm_arr, min_models, ppm_models, rate_history)

        sims = run_pts_simulation(sim_row, n_sims=n_sims)
        p_over = float(np.mean(sims > line))
        p_under = float(np.mean(sims < line))

        pred = sim_row["STAT_Q50"]
        side = "Over" if p_over >= p_under else "Under"
        p_side = p_over if side == "Over" else p_under
        hit = (side == "Over" and actual_pts > line) or (side == "Under" and actual_pts < line)
        edge = abs(pred - line)
        reccomended_bet = 1 if p_side >= prob_threshold else 0

        rows.append({
            "date": date,
            "player": player,
            "player_pts_mean": pdf_pts_mean,
            "pred points": round(pred, 2),
            "sim_mean": round(float(np.mean(sims)), 2),
            "side": side,
            "line": line,
            "actual points": actual_pts,
            "p_over": round(p_over, 3),
            "p_under": round(p_under, 3),
            "p_side": round(p_side, 3),
            "edge": round(edge, 2),
            "hit": int(hit),
            "reccomended_bet": reccomended_bet,
        })
    return pd.DataFrame(rows)

START, END = "2025-11-01", "2026-04-19"   # inclusive; tweak as needed

dates = sorted(
    p.stem for p in ODDS_DIR.glob("*.csv")
    if START <= p.stem <= END
)
print(f"Running {len(dates)} dates: {dates[0]} → {dates[-1]}")

np.random.seed(42)

all_res = []
for d in dates:
    r = backtest_one_date(
        d,
        pts_df=pts_df, min_df=min_df,
        min_models=min_models, ppm_models=ppm_models,
        min_feature_names=min_feature_names,
        ppm_feature_names=ppm_feature_names,
    )
    if not r.empty:
        all_res.append(r)

res = pd.concat(all_res, ignore_index=True) if all_res else pd.DataFrame()
print(f"Total bets: {len(res)}")
print(f"Total hit rate: {round(res['hit'].sum() / len(res), 3)}")

rec = res[res["reccomended_bet"] == 1]
if len(rec):
    print(f"Recommended bets (p_side >= {PROB_THRESHOLD}): {len(rec)} | hit rate: {round(rec['hit'].sum() / len(rec), 3)}")

# Hit rate by simulation-probability bucket
res["p_side_bucket"] = pd.cut(
    res["p_side"],
    bins=[0, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 1.0],
    include_lowest=True,
)
print("Hit rate by p_side bucket:")
print(res.groupby("p_side_bucket")["hit"].agg(["count", "mean"]).round(3))

res.sample(10)

Running 159 dates: 2025-11-01 → 2026-04-19
Total bets: 11867
Total hit rate: 0.507
Recommended bets (p_side >= 0.58): 5763 | hit rate: 0.518
Hit rate by p_side bucket:
               count   mean
p_side_bucket              
(-0.001, 0.5]     49  0.531
(0.5, 0.55]     4054  0.499
(0.55, 0.6]     3237  0.498
(0.6, 0.65]     2296  0.499
(0.65, 0.7]     1280  0.534
(0.7, 0.75]      608  0.539
(0.75, 1.0]      343  0.586


,date,player,player_pts_mean,pred points,sim_mean,side,line,actual points,p_over,p_under,p_side,edge,hit,reccomended_bet,p_side_bucket
8077,2026-02-22,Nolan Traore,8.910714,12.45,14.01,Over,10.5,11,0.640,0.360,0.640,1.95,1,1,"(0.6, 0.65]"
10609,2026-03-27,Collin Sexton,16.933852,15.43,18.33,Over,13.5,22,0.724,0.276,0.724,1.93,1,1,"(0.7, 0.75]"
8325,2026-02-26,Malik Monk,14.637681,13.21,14.80,Over,13.5,8,0.541,0.459,0.541,0.29,0,0,"(0.5, 0.55]"
11023,2026-04-01,Brice Sensabaugh,11.960674,19.61,23.07,Over,22.5,28,0.507,0.493,0.507,2.89,1,0,"(0.5, 0.55]"
5438,2026-01-14,Matas Buzelis,12.356688,18.20,20.08,Over,17.5,7,0.617,0.383,0.617,0.70,0,1,"(0.6, 0.65]"
8953,2026-03-06,Davion Mitchell,6.996622,8.09,8.26,Under,8.5,13,0.454,0.546,0.546,0.41,0,0,"(0.5, 0.55]"
10198,2026-03-21,Sam Merrill,8.899471,12.37,12.75,Under,12.5,15,0.498,0.501,0.501,0.13,0,0,"(0.5, 0.55]"
8595,2026-03-01,Jalen Duren,13.536232,21.99,25.10,Over,18.5,16,0.798,0.202,0.798,3.49,0,1,"(0.75, 1.0]"
10796,2026-03-29,Karl-Anthony Towns,21.907563,17.62,21.12,Over,17.5,15,0.671,0.329,0.671,0.12,0,1,"(0.65, 0.7]"
5663,2026-01-17,Ausar Thompson,9.594872,11.63,11.70,Over,10.5,7,0.518,0.482,0.518,1.13,0,0,"(0.5, 0.55]"
